# Sampling

We want to collect reviews wrote by active users, meaning :
- Users with more than 20 reviews
- A sample of 50,000 users among them 
- For active each user, collect all his reviews

| Métrique | Valeur |
|----------|--------|
| Reviews totales | ~27 millions |
| Utilisateurs uniques | 10 297 355 |
| Utilisateurs actifs (≥ 20 reviews) | 137 305 (1,33 % des utilisateurs) |
| Utilisateurs échantillonnés | 50 000 (36,4 % des actifs) |


| Itération | Outil | Reviews | Users | Volumétrie OK ? | Structure préservée ? | Bug notable |
|-----------|-------|---------|-------|-----------------|-----------------------|-------------|
| 1 | Streaming Python | 2 442 098 | 50 000 | Légèrement > 2M | Oui (verbatim) | — |
| 2 | Pandas chunked | ~2,4M | 50 000 | Légèrement > 2M | Oui (tous champs) | — |
| 3 | Polars | 2 425 431 | 50 000 | Légèrement > 2M | Oui | Filtre temporel = 0 (ms vs s) |
| 4 | Dask CPU | 2 415 706 | 50 000 | Légèrement > 2M | 9 colonnes sélectionnées | — |
| 5A | cuDF row-groups | ~2,44M | 50 000 | Légèrement > 2M | Oui (tous champs) | — |
| 5B | cuDF byte-range | 2 442 267 | 50 000 | Légèrement > 2M | Oui | — |
| 5C | cuDF + RMM | 2 442 267 | 50 000 | Légèrement > 2M | Oui | — |
| 5D | cuDF temporel | 200 000 | N/A | Non (trop faible) | Profils brisés | Timestamp ms/s + volume |
| 6 | Dask + cuDF | 2 420 879 | 50 000 | Légèrement > 2M | 8 colonnes sélectionnées | — |
| 7 | PySpark | Non exécuté | — | — | — | Sampling approximatif |
| 8 | DuckDB | 2 429 097 | 50 000 | Légèrement > 2M | Oui (tous champs) | Filtre temporel (ms vs s) |

### First things first ###
Convert the JSONL to Parquet first. Every subsequent read will be 10-50x faster:

In [ ]:
import polars as pl
import duckdb

# With Polars:
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books-polars.parquet")

# # With DuckDB:
# duckdb.sql("""
#     COPY (SELECT * FROM read_json_auto('data/Books.jsonl', format='newline_delimited'))
#     TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
# """)

# DuckDB handles messy JSONL flawlessly and writes Parquet very fast
con = duckdb.connect()
print("Converting JSONL → Parquet (this takes ~5-10 min)...")
con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'data/Books.jsonl',
            format='newline_delimited',
            maximum_object_size=10485760
        )
    ) TO 'data/Books-duckdb.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")
print("Done!")
con.close()

### Itération 5C : cuDF avec gestion mémoire RMM(selected) ###

In [ ]:
import cudf
import cupy as cp
import rmm
import gc
import random
import time

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

def print_memory_status(label=""):
    """Show current GPU memory usage."""
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    print(f"  [{label}] GPU: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB "
          f"(free: {info.free/1e9:.2f} GB)")
    pynvml.nvmlShutdown()


# ── Configure RMM memory pool ──────────────────────────────────
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=True,
)

print_memory_status("Before start")

# ================================================================
# PHASE 1: Load JSONL, count per user, find active users
# ================================================================
start = time.time()
print("Phase 1: Chargement en GPU memory...")

gdf = cudf.read_json(DATA_PATH, lines=True)
gdf['rating'] = gdf['rating'].astype('int8')

print_memory_status("After load")
print(f"  GPU DataFrame: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Comptage sur GPU
user_counts = gdf['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_REVIEWS].index

# ── Transfer active user IDs to CPU immediately ────────────────
active_list = active_users.to_pandas().tolist()
print(f"  Active users: {len(active_list):,}")

# ── Sample 50,000 randomly (CPU) ───────────────────────────────
random.seed(SEED)
selected_users = random.sample(active_list, min(NUM_USERS, len(active_list)))

# ── FLUSH: delete the counts, we only need the user ID list now ─
del user_counts, active_users, active_list
flush_memory()
print_memory_status("After count flush")

# ================================================================
# PHASE 2: Filter the full DataFrame for sampled users
# ================================================================
print("\nPhase 2: Filtrage...")

# Convert back to cudf Series for GPU-side isin()
selected_series = cudf.Series(selected_users)
mask = gdf['user_id'].isin(selected_series)
sample_gdf = gdf[mask]

print(f"  Reviews matched: {len(sample_gdf):,}")

# ── FLUSH: delete the full DataFrame — we no longer need it ────
del gdf, mask, selected_series
flush_memory()
print_memory_status("After filter flush")

# ================================================================
# PHASE 3: Transfer to CPU and save
# ================================================================
print("\nPhase 3: Transfert vers CPU et sauvegarde...")

sample = sample_gdf.to_pandas()

# ── FLUSH: delete the GPU DataFrame — data is on CPU now ───────
del sample_gdf
flush_memory()
print_memory_status("After GPU->CPU flush")

elapsed = time.time() - start
print(f"\nTemps d'execution: {elapsed:.2f}s")
print(f"Reviews echantillonnees: {len(sample):,}")
print(f"Utilisateurs uniques: {sample['user_id'].nunique():,}")

# Save
sample.to_parquet('sample-cudf-claude/sample_gpu_active_users.parquet', compression='snappy')

# ── FINAL FLUSH: free everything including the pandas DataFrame ─
del sample
gc.collect()
print_memory_status("Final cleanup")

Résultat : 2 442 267 reviews, 50 000 utilisateurs — Temps d'exécution : 10,03 s
#### Représentativité
- Cette itération applique la même stratégie d'échantillonnage par utilisateurs actifs que les itérations 1 à 5B : comptage global des reviews par user_id, seuil ≥ 20 reviews, tirage aléatoire uniforme de 50 000 utilisateurs (random.seed(42)), puis collecte exhaustive de toutes leurs reviews. La logique statistique est strictement identique ; seul le moteur d'exécution diffère.
- Le fichier JSONL complet (~27M reviews, ~16 Go) est chargé intégralement en mémoire GPU d'un seul bloc via cudf.read_json(DATA_PATH, lines=True). Aucune ligne n'est ignorée lors du chargement, ce qui garantit un comptage exhaustif et sans biais des 10 297 355 utilisateurs uniques.
- L'activation de la mémoire unifiée CUDA (rmm.reinitialize(managed_memory=True, pool_allocator=True)) permet au gestionnaire de mémoire RMM de déborder transparentement de la VRAM vers la RAM système si nécessaire. Cela élimine le risque d'un OutOfMemoryError GPU qui tronquerait silencieusement les données, renforçant la fiabilité du comptage.
- Le résultat (137 305 utilisateurs actifs, 2 442 267 reviews échantillonnées) est identique aux itérations 5A et 5B, confirmant la reproductibilité de la stratégie indépendamment de l'implémentation technique.
#### Volumétrie cible
- 2 442 267 reviews — légèrement au-dessus de la borne supérieure de la fourchette cible 500K–2M. Ce dépassement est un compromis conscient et justifié : la collecte de toutes les reviews de chaque utilisateur sélectionné est nécessaire pour préserver l'intégralité des profils utilisateurs. Tronquer arbitrairement les reviews d'un utilisateur biaiserait toute analyse comportementale (évolution des notes, diversité des produits, fréquence de contribution).
- Pour ramener le volume strictement dans la cible, deux leviers sont disponibles sans altérer la logique :
    - Réduire NUM_USERS à ~35 000–42 000 (volume estimé ~1,7M–2,0M).
    - Appliquer un filtre temporel post-échantillonnage (par exemple 2020–2023), ce qui réduit le volume tout en conservant les profils complets sur la période retenue.
- Le temps d'exécution de 10 secondes (contre ~15 min pour le streaming Python) démontre que l'accélération GPU ne compromet ni le volume ni la qualité de l'échantillon.
#### Préservation de la structure des données
- Le DataFrame GPU (gdf) contient l'intégralité des colonnes du JSONL source : user_id, parent_asin, asin, rating, timestamp, title, text, helpful_vote, verified_purchase, images, etc. Aucune projection (columns=[...]) n'est appliquée.
- La seule transformation de type est rating → int8, qui est sans perte puisque les notes Amazon sont des entiers dans l'intervalle [1, 5]. Cette optimisation réduit l'empreinte VRAM de la colonne d'un facteur 8 (64 bits → 8 bits) sans altérer les valeurs.
- Le pipeline préserve les trois niveaux de structure du dataset :
    - Utilisateur → reviews : complet pour chaque utilisateur sélectionné (toutes ses reviews sont incluses, pas d'échantillonnage intra-utilisateur).
    - Review → produit : le lien parent_asin est conservé, permettant des analyses produit-centriques (notes moyennes, nombre de reviewers par produit, etc.).
    - Dimension temporelle : les timestamp originaux sont préservés sans conversion, permettant des analyses de séries temporelles et des filtres post-hoc sur n'importe quelle période.
- Le protocole de gestion mémoire explicite (flush_memory() avec gc.collect() + libération du memory pool CuPy + synchronisation CUDA) et le monitoring VRAM (print_memory_status()) à chaque étape sont des garanties opérationnelles : ils assurent que les transferts GPU→CPU (to_pandas()) se font sans corruption due à la pression mémoire, et fournissent une trace d'audit de la consommation à chaque phase du pipeline.

### Itération 5D : cuDF stratifié temporel(selected) ###

In [4]:
import cudf
import cupy as cp
import rmm
import random
import gc
import pandas

DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42
TARGET_TOTAL = 2_000_000
TARGET_YEARS = [2020, 2021, 2022, 2023]


# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    """
    flush_memory()
    print("Chargement en GPU memory...")
    monitor_gpu_memory()
    gdf = cudf.read_json(DATA_PATH, lines=True)
    monitor_gpu_memory()
    
    gdf['rating'] = gdf['rating'].astype('int8')

    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'], unit='ms')
    gdf['year'] = gdf['timestamp'].dt.year

    # ── Year-by-year diagnostics ──────────────────────────────────
    gdf_target = gdf[gdf['year'].isin(TARGET_YEARS)]
    year_stats = (
        gdf_target.groupby('year')
        .agg({
            'user_id': ['count', 'nunique'],
            'rating': 'mean'
        })
    )
    # Transfer to pandas for display
    ys = year_stats.to_pandas()
    ys.columns = ['review_count', 'unique_users', 'avg_rating']
    ys = ys.sort_index()
    print(f"\n── Year-by-Year Breakdown ({TARGET_YEARS[0]}–{TARGET_YEARS[-1]}) ──")
    print(ys.to_string())
    print(f"\nTotal reviews: {ys['review_count'].sum():,}")
    print(f"Total unique users (with overlap): {ys['unique_users'].sum():,}")

    # ① Restrict to target period
    gdf_period = gdf[gdf['year'].isin(TARGET_YEARS)]
    del gdf
    monitor_gpu_memory()

    print(f"Reviews in {TARGET_YEARS[0]}–{TARGET_YEARS[-1]}: {len(gdf_period):,}")

    # ② Count reviews per user within the period
    user_counts = gdf_period['user_id'].value_counts().reset_index()
    user_counts.columns = ['user_id', 'review_count']

    # ③ Keep only users with >= MIN_REVIEWS in this period
    active_in_period = user_counts[user_counts['review_count'] >= MIN_REVIEWS]
    print(f"Active users (>= {MIN_REVIEWS} reviews in period): {len(active_in_period):,}")

    # ④ Sample up to 50,000 users
    active_list = active_in_period['user_id'].to_pandas().tolist()
    random.seed(SEED)
    n_to_sample = min(NUM_USERS, len(active_list))
    sampled_users = random.sample(active_list, n_to_sample)
    print(f"Sampled users: {n_to_sample:,} / {len(active_list):,}")

    # ⑤ Collect ALL their reviews in the period (no volume cap)
    sampled_series = cudf.Series(sampled_users)
    sample_gdf = gdf_period[gdf_period['user_id'].isin(sampled_series)]
    monitor_gpu_memory()

    sample = sample_gdf.to_pandas()
    print(f"Reviews: {len(sample):,} from {sample['user_id'].nunique():,} users")
    print(f"Avg reviews/user: {len(sample) / sample['user_id'].nunique():.1f}")
    del sample_gdf, gdf_period
    flush_memory()
    monitor_gpu_memory()  # after transferring to CPU and freeing GPU
    
    return sample
    
# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

# Utilisation
if __name__ == '__main__':
    import time
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-tempor/sample_gpu_temporal.parquet', compression='snappy')

Chargement en GPU memory...
GPU Memory: 1.67/34.19 GB
GPU Memory: 34.19/34.19 GB
GPU memory utilisée: 16.04 GB

── Year-by-Year Breakdown (2020–2023) ──
      review_count  unique_users  avg_rating
year                                        
2020       1787771       1129095    4.433200
2021       1881741       1174589    4.426835
2022       1667582       1022383    4.414456
2023        651169        440007    4.425020

Total reviews: 5,988,263
Total unique users (with overlap): 3,766,074
GPU Memory: 34.04/34.19 GB
Reviews in 2020–2023: 5,988,263
Active users (>= 20 reviews in period): 18,097
Sampled users: 18,097 / 18,097
GPU Memory: 34.15/34.19 GB
Reviews: 856,620 from 18,097 users
Avg reviews/user: 47.3
GPU Memory: 34.15/34.19 GB

⚡ Temps d'exécution GPU: 16.82s
   Reviews échantillonnées: 856,620


Résultat : 856,620 reviews (échantillonnage stratifié proportionnel, période 2020–2023)
#### Représentativité
- Cette cellule adopte une stratégie d'échantillonnage fondamentalement différente des itérations précédentes. Au lieu de sélectionner des utilisateurs actifs puis de collecter toutes leurs reviews, elle réalise un échantillonnage stratifié par année sur la période 2020–2023.
- La distribution proportionnelle (proportion = len(year_data) / total_in_period) garantit que chaque année contribue à l'échantillon au prorata de son poids réel dans le dataset sur la période ciblée. Une année avec 40 % des reviews de la période fournira ~200K reviews sur les 500K, ce qui préserve fidèlement la dynamique temporelle du marché (croissance, saisonnalité, etc.).
- Ce choix est particulièrement pertinent pour les analyses de tendances temporelles (évolution des sentiments, émergence de thèmes, variation de la notation moyenne par année) où une sur- ou sous-représentation d'une année fausserait les conclusions.
- La conversion des timestamps est correctement paramétrée (cudf.to_datetime(gdf['timestamp'], unit='ms')) puisque les données Amazon utilisent des timestamps en millisecondes depuis l'époque Unix. Cela permet un découpage annuel fiable.
- Limite : cette approche ne préserve pas les profils utilisateurs complets — un utilisateur peut n'avoir qu'une fraction de ses reviews dans l'échantillon. Le tirage aléatoire intra-année (year_data.sample(n=..., random_state=SEED)) est reproductible mais ne garantit pas la couverture intégrale d'un utilisateur donné.
#### Volumétrie cible
- TARGET_TOTAL = 500_000 place l'échantillon exactement à la borne inférieure de la fourchette cible 500K–2M. Ce volume est un compromis volontaire :
- Suffisamment grand pour des analyses statistiquement robustes sur 4 années.
- Suffisamment compact pour permettre des traitements interactifs (exploration, visualisation, entraînement de modèles légers) sans contrainte mémoire excessive.
-La répartition proportionnelle avec récupération du reste sur la dernière année (remaining) garantit que le total atteint exactement 500K, sans sur-échantillonnage ni perte.
- Pour augmenter la volumétrie (par exemple 1M), il suffit de modifier TARGET_TOTAL = 1_000_000 ; la logique proportionnelle s'adapte automatiquement.
#### Préservation de la structure des données
- Chaque review conserve l'intégralité de ses champs (user_id, parent_asin, rating, timestamp, text, title, helpful_vote, verified_purchase, etc.) — aucune projection ni troncature de colonnes.
- Le filtrage temporel (gdf['year'].isin([2020, 2021, 2022, 2023])) exclut volontairement les reviews antérieures à 2020, ce qui réduit la profondeur historique mais concentre l'analyse sur les données les plus récentes et pertinentes.
- La relation review → produit (parent_asin) est préservée, permettant des analyses produit-centriques (distribution des notes par produit, diversité des reviewers par produit).
- En revanche, la relation utilisateur → ensemble complet de ses reviews est partiellement brisée : un utilisateur ayant écrit des reviews en 2019 et 2021 n'aura dans l'échantillon que celles de 2021, et uniquement si elles ont été tirées au sort. Ce compromis est inhérent à l'échantillonnage temporel stratifié et acceptable pour des analyses centrées sur les tendances plutôt que sur les profils utilisateurs.


# Cleaning up memory #

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
import gc

# 1. GPU cleanup (safe even if no GPU libs loaded)
try:
    import cupy as cp
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()
except ImportError:
    pass

try:
    import rmm
    rmm.reinitialize()
except (ImportError, RuntimeError):
    pass

try:
    import torch
    torch.cuda.empty_cache()
except ImportError:
    pass

# 2. Close DuckDB connections
try:
    import duckdb
    con.close()       # adjust to your connection variable name
except:
    pass

# 3. Nuke user-defined variables
_keep = {'In', 'Out', 'get_ipython', 'exit', 'quit', 'open'}
for _name in list(globals()):
    if not _name.startswith('_') and _name not in _keep:
        del globals()[_name]

# 4. Force GC
import gc
gc.collect()